# Florence-2 V6 — Inferencia + Demo Gradio

Notebook de pruebas para el modelo **V6** (Florence-2-base full fine-tune sobre `Lacax/Tickets-total`).
Cubre H5 del plan V6: demo Gradio + verificación cruzada OCR.space sobre el crop del bbox.

## Diferencias respecto a V5

1. **Modelo**: Florence-2-base (~230M) en lugar de DeepSeek-VL + LoRA.
2. **Checkpoint**: full fine-tune en Drive (`/content/drive/MyDrive/TFG/V6_checkpoints/h3_full_ft_best/`). No hay adapter.
3. **Tarea**: tag custom `<EXTRACT_TOTAL>` → `total<loc_x1><loc_y1><loc_x2><loc_y2>` (un único campo escalar + bbox).
4. **Verificación cruzada**: OCR.space sobre el crop del bbox confirma o no el `total` predicho.
5. **Stack independiente de V5**: `transformers>=4.41,<4.46`, sin Unsloth, sin flash-attn (T4 free).

Compatible con Google Colab (T4 free). Tiempo aprox: 2–4 s por ticket.

### Celda 0 — Setup (instala deps; reinicia el kernel a mano)

**IMPORTANTE**: Colab trae `transformers>=4.50` preinstalado y Florence-2 solo funciona con `<4.46`. La celda hace el downgrade pero **NO reinicia el kernel**.

Cuando termine: **Entorno de ejecución → Reiniciar entorno de ejecución** (o `Ctrl+M .`). Tras el restart, salta esta celda y arranca por la celda 1.

In [ ]:
# [CELDA 0] SETUP — solo instala deps. Reinicia el kernel manualmente al terminar.
# Florence-2 requiere transformers<4.46. Colab trae >=4.50, forzamos downgrade.
!pip install -q --force-reinstall --no-deps transformers==4.45.2
!pip install -q accelerate einops timm huggingface_hub pillow gradio requests

print('=' * 60)
print('Deps instaladas.')
print('AHORA: Entorno de ejecución -> Reiniciar entorno de ejecución')
print('Luego ejecuta desde la celda 1.')
print('=' * 60)

### Celda 1 — Mount Drive (donde vive el best checkpoint de H3)

In [ ]:
# [CELDA 1] Mount Drive
from google.colab import drive
import os

drive.mount('/content/drive')
BEST_DIR = '/content/drive/MyDrive/TFG/V6_checkpoints/h3_full_ft_best'
assert os.path.exists(BEST_DIR), f'no encuentro {BEST_DIR} — revisa la ruta del checkpoint'
print('contenido del best:', os.listdir(BEST_DIR))

### Celda 2 — Cargar Florence-2 V6 (best checkpoint H3)

El checkpoint guardado por `Trainer.save_model` incluye el `config.json` con `auto_map` apuntando al código custom de Florence-2 — basta `trust_remote_code=True`. Tras el load llamamos `tie_weights()` para mitigar el bug de tied weights de Florence-2 + safetensors.

In [ ]:
# [CELDA 2] Carga del modelo V6
import transformers, torch
assert transformers.__version__.startswith('4.45'), (
    f'transformers={transformers.__version__} pero Florence-2 requiere 4.45.x — vuelve a la celda 0'
)
from transformers import AutoModelForCausalLM, AutoProcessor

model = AutoModelForCausalLM.from_pretrained(
    BEST_DIR,
    trust_remote_code=True,
    torch_dtype=torch.float16,
).to('cuda').eval()
model.tie_weights()

processor = AutoProcessor.from_pretrained(BEST_DIR, trust_remote_code=True)

EXTRACT_TAG = '<EXTRACT_TOTAL>'
tag_ids = processor.tokenizer(EXTRACT_TAG, add_special_tokens=False).input_ids
print(f"OK V6 — dtype={model.dtype}, device={next(model.parameters()).device}")
print(f"Tag '{EXTRACT_TAG}' tokenized -> {tag_ids} (debe ser 1 id único)")
vram = torch.cuda.memory_allocated() / 1e9
print(f'VRAM: {vram:.2f} GB')

### Celda 3 — Utilidades (parse, de-cuantización, `run_inference`)

Replica los helpers del notebook de training (celdas R/S de `V6_Florence2_Total.ipynb`). De-cuantización con midpoint del bin para compensar el `floor` del quantize.

In [ ]:
# [CELDA 3] Utilidades
import re
from PIL import Image

NUM_BBOX_BINS = 1000

OUTPUT_RE = re.compile(
    r'(?P<total>-?\d+(?:\.\d+)?)<loc_(?P<x1>\d+)><loc_(?P<y1>\d+)><loc_(?P<x2>\d+)><loc_(?P<y2>\d+)>'
)

def parse_output(text):
    m = OUTPUT_RE.search(text)
    if not m:
        return None
    try:
        total = float(m.group('total'))
    except ValueError:
        return None
    loc = tuple(int(m.group(k)) for k in ('x1', 'y1', 'x2', 'y2'))
    if not all(0 <= v <= 999 for v in loc):
        return None
    return total, loc

def dequantize_bbox(loc, image_size, num_bins=NUM_BBOX_BINS):
    W, H = image_size
    x1, y1, x2, y2 = loc
    return (
        (x1 + 0.5) * (W / num_bins),
        (y1 + 0.5) * (H / num_bins),
        (x2 + 0.5) * (W / num_bins),
        (y2 + 0.5) * (H / num_bins),
    )

def run_inference(image: Image.Image):
    """Devuelve (raw_text, parsed_total, bbox_px) o (raw_text, None, None) si malformado."""
    inputs = processor(text=EXTRACT_TAG, images=image, return_tensors='pt').to('cuda')
    inputs['pixel_values'] = inputs['pixel_values'].to(model.dtype)
    with torch.inference_mode():
        out = model.generate(
            input_ids=inputs['input_ids'],
            pixel_values=inputs['pixel_values'],
            max_new_tokens=20,
            do_sample=False,
            num_beams=1,
        )
    raw = processor.batch_decode(out, skip_special_tokens=False)[0]
    parsed = parse_output(raw)
    if parsed is None:
        return raw, None, None
    total, loc = parsed
    bbox_px = dequantize_bbox(loc, image.size)
    return raw, total, bbox_px

print('Utilidades cargadas.')

### Celda 4 — Test rápido sobre una imagen

Sube una imagen al directorio de Colab y pon su nombre en `IMG`.

In [ ]:
# [CELDA 4] Test rápido — un solo ticket
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

IMG = '/content/img.png'  # <-- cambia por tu ticket

image = Image.open(IMG).convert('RGB')
raw, total, bbox = run_inference(image)

print('=' * 60)
print(f'Imagen: {IMG} · size={image.size}')
print('=' * 60)
print('Raw output:', repr(raw))
if total is None:
    print('-> JSON/output INVÁLIDO')
else:
    print(f'-> total predicho = {total}')
    print(f'-> bbox píxeles   = {tuple(round(v,1) for v in bbox)}')
    fig, ax = plt.subplots(figsize=(8, 10))
    ax.imshow(image)
    px1, py1, px2, py2 = bbox
    ax.add_patch(Rectangle((px1, py1), px2-px1, py2-py1,
                           fill=False, edgecolor='red', linewidth=3))
    ax.set_title(f'pred total = {total}')
    ax.axis('off')
    plt.show()

### Celda 5 — Verificación cruzada con OCR.space

Pipeline: bbox → crop con padding → OCR.space → extraer números del texto → comparar con `total` predicho.

**Veredictos:**
- ✅ **MATCH**: el total predicho aparece en el OCR del crop (tolerancia ±0.01).
- ⚠️ **MISMATCH**: el OCR ve un número distinto en la zona — Florence-2 puede haber alucinado un dígito.
- ❓ **NO_TEXT**: OCR.space no extrae texto del crop (bbox demasiado pequeño o imagen de baja calidad).

API key gratuita en https://ocr.space/ocrapi (`helloworld` funciona pero limitada).

In [ ]:
# [CELDA 5] Verificación cruzada OCR.space
import requests, io, re

OCR_SPACE_KEY = 'helloworld'  # <-- sustituye por tu key real para evitar rate limits

NUMBER_RE = re.compile(r'-?\d+[.,]\d{1,2}')

def crop_with_padding(image: Image.Image, bbox, pad_ratio=0.15):
    W, H = image.size
    x1, y1, x2, y2 = bbox
    bw, bh = x2 - x1, y2 - y1
    px, py = bw * pad_ratio, bh * pad_ratio
    return image.crop((
        max(0, int(x1 - px)), max(0, int(y1 - py)),
        min(W, int(x2 + px)), min(H, int(y2 + py)),
    ))

def ocr_space_crop(crop: Image.Image, api_key=OCR_SPACE_KEY, language='spa'):
    buf = io.BytesIO()
    crop.save(buf, format='PNG')
    buf.seek(0)
    try:
        r = requests.post(
            'https://api.ocr.space/parse/image',
            files={'file': ('crop.png', buf, 'image/png')},
            data={'apikey': api_key, 'language': language, 'OCREngine': 2, 'scale': 'true'},
            timeout=30,
        )
        data = r.json()
    except Exception as e:
        return '', f'error: {e}'
    if data.get('IsErroredOnProcessing'):
        return '', f"ocr error: {data.get('ErrorMessage')}"
    parsed = data.get('ParsedResults') or []
    text = parsed[0].get('ParsedText', '') if parsed else ''
    return text, None

def cross_validate(image: Image.Image, pred_total: float, bbox, tol=0.01):
    crop = crop_with_padding(image, bbox)
    text, err = ocr_space_crop(crop)
    if err:
        return {'verdict': '❓ NO_TEXT', 'ocr_text': '', 'numbers': [], 'error': err, 'crop': crop}
    numbers = []
    for s in NUMBER_RE.findall(text):
        try:
            numbers.append(float(s.replace(',', '.')))
        except ValueError:
            pass
    if not numbers:
        verdict = '❓ NO_TEXT'
    elif any(abs(n - pred_total) <= tol for n in numbers):
        verdict = '✅ MATCH'
    else:
        verdict = '⚠️ MISMATCH'
    return {'verdict': verdict, 'ocr_text': text, 'numbers': numbers, 'error': None, 'crop': crop}

# Sanity sobre la imagen de la celda 4 (si quedó cargada)
if 'total' in dir() and total is not None:
    res = cross_validate(image, total, bbox)
    print('verdict :', res['verdict'])
    print('ocr text:', repr(res['ocr_text']))
    print('numbers :', res['numbers'])

### Celda 6 — Demo Gradio

Pipeline completo: subir ticket → Florence-2 (total + bbox) → crop → OCR.space → veredicto. URL pública con `share=True` (gradio.live, 72 h).

In [ ]:
# [CELDA 6] Gradio demo V6
import gradio as gr
import numpy as np
from PIL import ImageDraw

def render_with_bbox(image: Image.Image, bbox, color='red', width=4):
    out = image.copy()
    draw = ImageDraw.Draw(out)
    x1, y1, x2, y2 = bbox
    draw.rectangle([x1, y1, x2, y2], outline=color, width=width)
    return out

def predict(image, ocr_api_key):
    if image is None:
        return None, 'Sin imagen', None, '', ''
    if not isinstance(image, Image.Image):
        image = Image.fromarray(image).convert('RGB')
    else:
        image = image.convert('RGB')

    raw, total, bbox = run_inference(image)
    if total is None:
        return image, '❌ Output mal-formado', None, raw, 'No se intentó verificación cruzada (sin bbox).'

    annotated = render_with_bbox(image, bbox, color='red')

    key = (ocr_api_key or '').strip() or OCR_SPACE_KEY
    crop = crop_with_padding(image, bbox)
    text, err = ocr_space_crop(crop, api_key=key)
    if err:
        verdict = f'❓ NO_TEXT ({err})'
        ocr_block = err
    else:
        numbers = []
        for s in NUMBER_RE.findall(text):
            try:
                numbers.append(float(s.replace(',', '.')))
            except ValueError:
                pass
        if not numbers:
            verdict = '❓ NO_TEXT (OCR no extrajo números)'
        elif any(abs(n - total) <= 0.01 for n in numbers):
            verdict = f'✅ MATCH — pred {total} aparece en el crop'
        else:
            verdict = f'⚠️ MISMATCH — pred {total} ; OCR ve {numbers}'
        ocr_block = text

    summary = (
        f'Total predicho: **{total}**\n\n'
        f'Bbox (px): {tuple(round(v,1) for v in bbox)}\n\n'
        f'Verificación cruzada OCR.space: {verdict}'
    )
    return annotated, summary, crop, raw, ocr_block

with gr.Blocks(title='Florence-2 V6 — Total extraction') as demo:
    gr.Markdown(
        '# DeepSeek/Florence-2 V6 — Extracción del `total`\n'
        'Modelo: Florence-2-base full fine-tune (eval_loss 1.398, dataset golden 130 tickets).\n'
        'Verificación cruzada con OCR.space sobre el crop del bbox predicho.'
    )
    with gr.Row():
        with gr.Column():
            inp = gr.Image(label='Sube un ticket', type='pil')
            key = gr.Textbox(label='OCR.space API key (opcional)', placeholder='helloworld por defecto', type='password')
            btn = gr.Button('Extraer total', variant='primary')
        with gr.Column():
            out_img = gr.Image(label='Ticket con bbox predicho')
            out_md = gr.Markdown(label='Resultado')
            out_crop = gr.Image(label='Crop enviado a OCR.space')
            out_raw = gr.Textbox(label='Output crudo Florence-2')
            out_ocr = gr.Textbox(label='Texto OCR.space sobre el crop', lines=4)
    btn.click(predict, inputs=[inp, key], outputs=[out_img, out_md, out_crop, out_raw, out_ocr])

demo.launch(share=True)